In [29]:
# TODO: [x] create groups of dbs, controllers
# TODO: [x] consolidate executors to calculate full performance
# TODO: [x] separate controllers into 2 trading pairs, one long other short
# TODO: [x] calculate short metrics
# TODO: [ ] (optional) get long/short epiphany
# TODO: [x] isolate config for each controller for metrics
# TODO: [wip] calculate general summarized metrics

In [30]:
import os
import sys
from typing import Dict, Any

import pandas as pd
import warnings
import logging
from dotenv import load_dotenv

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

from core.data_sources.hummingbot_database import HummingbotDatabase
import research_notebooks.statarb_v2.stat_arb_performance_utils as utils

logging.getLogger("asyncio").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")
load_dotenv()

FETCH_DBS = False
if FETCH_DBS:
    utils.fetch_dbs(root_path)

dbs = [db_path for db_path in os.listdir(os.path.join(root_path, "data", "live_bot_databases")) if db_path != ".gitignore"]
dbs

['binance_perpetual-2025-11-2-0130-2025.sqlite',
 'binance_perpetual-2025-11-2-0150-2025.sqlite',
 'binance_perpetual-2025-11-2-0000-2025.sqlite',
 'binance_perpetual-2025-11-2-0240-2025.sqlite',
 'okx_perpetual-ts-9-02-2025.sqlite',
 'binance_perpetual-2025-11-1-2000-2025.sqlite',
 'binance_perpetual-2025-11-1-1830-2025.sqlite',
 'binance_perpetual-2025-11-1-2300-2025.sqlite',
 'binance_perpetual-2025-11-2-0420-2025.sqlite']

In [31]:
all_controllers = []
all_executors = pd.DataFrame()
for db_name in dbs:
    try:
        db = HummingbotDatabase(db_name=db_name, root_path=root_path)
        executors_df = db.get_executors_data()
        executors_df["db_name"] = db_name
        controllers = db.get_controller_data().to_dict(orient="records")
        valid_controllers = [controller for controller in controllers if controller["config"]["controller_name"] == "stat_arb"]
        valid_controllers_ids = [controller["id"] for controller in valid_controllers]
        all_controllers.extend(valid_controllers)
        all_executors = pd.concat([all_executors, executors_df[executors_df["controller_id"].isin(valid_controllers_ids)]])
    except Exception as e:
        print(e)
        continue
all_configs = {controller["id"]: controller for controller in all_controllers}
print(f"Total controller configs: {len(all_configs)}")

Total controller configs: 11


In [32]:
all_trades = []

for _, executor in all_executors.iterrows():
    custom_info = executor["custom_info"]
    for order_filled in custom_info["filled_orders"]:
        for _, fill in order_filled["order_fills"].items():

            trade_type = order_filled["trade_type"]  # BUY or SELL
            position_action = order_filled["position"]  # OPEN or CLOSE

            position_multiplier = 1 if (trade_type == "BUY" and position_action == "OPEN") or (trade_type == "SELL" and position_action == "CLOSE") else -1
            fill_dict = {
                "db_name": executor["db_name"],
                "controller_id": executor["controller_id"],
                "side": executor["config"]["side"],
                "trading_pair": order_filled["trading_pair"],
                "order_type": order_filled["order_type"],
                "trade_type": trade_type,
                "cumulative_fee_paid_quote": sum([float(flat_fee["amount"]) for flat_fee in fill["fee"]["flat_fees"]]),  # this will be only valid when percent_token = USDT
                "position_action": order_filled["position"],
                "timestamp": utils.ensure_timestamp_in_seconds(fill["fill_timestamp"]),
                "price": float(fill["fill_price"]),
                "base_amount": float(fill["fill_base_amount"]),
                "quote_amount": float(fill["fill_quote_amount"]),
                "position_multiplier": position_multiplier
            }
            all_trades.append(fill_dict)
all_trades_df = pd.DataFrame(all_trades)
all_trades_df

,db_name,controller_id,side,trading_pair,order_type,trade_type,cumulative_fee_paid_quote,position_action,timestamp,price,base_amount,quote_amount,position_multiplier
0,binance_perpetual-2025-11-2-0130-2025.sqlite,binance-perpetual||MELANIA-USDT||IP-USDT||2025...,1,MELANIA-USDT,LIMIT,BUY,0.00191665,OPEN,1741668182,0.699,13.71,9.58329,1
1,binance_perpetual-2025-11-2-0130-2025.sqlite,binance-perpetual||MELANIA-USDT||IP-USDT||2025...,1,MELANIA-USDT,MARKET,SELL,0.00481221,CLOSE,1741668187,0.702,13.71,9.62442,1
2,binance_perpetual-2025-11-2-0130-2025.sqlite,binance-perpetual||MELANIA-USDT||IP-USDT||2025...,1,MELANIA-USDT,LIMIT,BUY,0.0047713,OPEN,1741668716,0.693,13.77,9.54261,1
3,binance_perpetual-2025-11-2-0130-2025.sqlite,binance-perpetual||MELANIA-USDT||IP-USDT||2025...,1,MELANIA-USDT,LIMIT,BUY,0.0019167,OPEN,1741668226,0.698,13.73,9.58354,1
4,binance_perpetual-2025-11-2-0130-2025.sqlite,binance-perpetual||MELANIA-USDT||IP-USDT||2025...,1,MELANIA-USDT,LIMIT,BUY,0.00191665,OPEN,1741668282,0.699,13.71,9.58329,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1584,binance_perpetual-2025-11-2-0420-2025.sqlite,binance-perpetual||ARB-USDT||CRV-USDT||2025||i...,2,CRV-USDT,LIMIT,SELL,0.00396675,OPEN,1741677754,0.387,20.5,7.9335,-1
1585,binance_perpetual-2025-11-2-0420-2025.sqlite,binance-perpetual||ARB-USDT||CRV-USDT||2025||i...,2,CRV-USDT,LIMIT,SELL,0.00158304,OPEN,1741679150,0.388,20.4,7.9152,-1
1586,binance_perpetual-2025-11-2-0420-2025.sqlite,binance-perpetual||ARB-USDT||CRV-USDT||2025||i...,2,CRV-USDT,LIMIT,SELL,0.0015834,OPEN,1741680317,0.39,20.3,7.917,-1
1587,binance_perpetual-2025-11-2-0420-2025.sqlite,binance-perpetual||ARB-USDT||CRV-USDT||2025||i...,2,CRV-USDT,LIMIT,SELL,0.00157964,OPEN,1741685264,0.391,20.2,7.8982,-1


In [33]:
print(f"Available controllers:\n" + "\n".join(f"   - {controller}" for controller in all_trades_df["controller_id"].unique()))
print("Copy and paste one in below cell to continue")

Available controllers:
   - binance-perpetual||MELANIA-USDT||IP-USDT||2025||isoweek11_2-0130
   - binance-perpetual||LINK-USDT||RED-USDT||2025||isoweek11_2-0150
   - binance-perpetual||OP-USDT||KAITO-USDT||2025||isoweek11_2-0000
   - binance-perpetual||VIRTUAL-USDT||ARC-USDT||2025||isoweek11_2-0240
   - okx_perpetual||NOT-USDT||AIXBT-USDT||2025||isoweek9_02
   - binance-perpetual||RUNE-USDT||IP-USDT||2025||isoweek11_1-2000
   - binance-perpetual||ETHFI-USDT||RED-USDT||2025||isoweek11_1-2000
   - binance-perpetual||NEIRO-USDT||SHELL-USDT||2025||isoweek11_1-1830
   - binance-perpetual||SEI-USDT||COW-USDT||2025||isoweek11_1-1830
   - binance-perpetual||ARB-USDT||CRV-USDT||2025||isoweek11_1-2300
   - binance-perpetual||ARB-USDT||CRV-USDT||2025||isoweek11_2-0420
Copy and paste one in below cell to continue


In [34]:
selected_controller = "okx_perpetual||NOT-USDT||AIXBT-USDT||2025||isoweek9_02"

## Plot general performance

In [35]:
import plotly.graph_objects as go

long_df = utils.calculate_performance_metrics(all_trades_df, selected_controller, 1)
long_performance_fig = await utils.plot_candles_with_global_pnl_chart(long_df)
long_performance_fig.write_image("long_performance.jpg", format="jpg", scale=3)

In [36]:
short_df = utils.calculate_performance_metrics(all_trades_df, selected_controller, 2)
short_performance_fig = await utils.plot_candles_with_global_pnl_chart(short_df, side=2)
short_performance_fig.write_image("short_performance.jpg", format="jpg", scale=3)

In [37]:


executors_data = db.get_executors_data()
controller_data = db.get_controller_data()

# top_value: float = None, bottom_value: float = None
config = executors_data.loc[0,'config']
top_value = config['end_price']
bottom_value = config['start_price']

long_performance_fig = await utils.plot_candles_with_global_pnl_chart(long_df, top_value = top_value, bottom_value = bottom_value)
long_performance_fig.write_image("long_performance.jpg", format="jpg", scale=3)

In [38]:
config = executors_data.loc[1,'config']
top_value = config['end_price']
bottom_value = config['start_price']

short_performance_fig = await utils.plot_candles_with_global_pnl_chart(short_df, top_value = top_value, bottom_value = bottom_value,side=2)
short_performance_fig.write_image("short_performance.jpg", format="jpg", scale=3)

In [39]:
config

{'id': '4C8JEi7eiAekqe5TmXRUuoubZ8pjnAWqB7A97ztAH5Xe',
 'type': 'grid_executor',
 'timestamp': 1741677754.7029836,
 'controller_id': 'binance-perpetual||ARB-USDT||CRV-USDT||2025||isoweek11_2-0420',
 'connector_name': 'binance_perpetual',
 'trading_pair': 'CRV-USDT',
 'start_price': 0.34818150650222224,
 'end_price': 0.4278184934977778,
 'limit_price': 0.44772774024666673,
 'side': 2,
 'total_amount_quote': 500.0,
 'min_spread_between_orders': 0.0004,
 'min_order_amount_quote': 7.5,
 'max_open_orders': 3,
 'max_orders_per_batch': 1,
 'order_frequency': 5,
 'activation_bounds': 0.0003,
 'safe_extra_spread': 0.0002,
 'triple_barrier_config': {'stop_loss': 0.1,
  'take_profit': 0.0008,
  'time_limit': 259200,
  'trailing_stop': {'activation_price': 0.03, 'trailing_delta': 0.005},
  'open_order_type': 2,
  'take_profit_order_type': 1,
  'stop_loss_order_type': 1,
  'time_limit_order_type': 1},
 'leverage': 50,
 'level_id': None,
 'deduct_base_fees': False,
 'keep_position': False,
 'coerce_

In [40]:
# Initialize clients
from core.data_sources.clob import CLOBDataSource
clob = CLOBDataSource()
from core.services.mongodb_client import MongoClient
from core.services.backend_api_client import BackendAPIClient

# mongo_client = MongoClient(
#     username=os.getenv("MONGO_INITDB_ROOT_USERNAME", "admin"),
#     password=os.getenv("MONGO_INITDB_ROOT_PASSWORD", "admin"),
#     host=os.getenv("MONGO_HOST", "localhost"),
#     port=os.getenv("MONGO_PORT", 27017),
#     database=os.getenv("MONGO_DATABASE", "quants_lab")
# )

uri = f"mongodb://{os.getenv('MONGO_INITDB_ROOT_USERNAME', 'admin')}:{os.getenv('MONGO_INITDB_ROOT_PASSWORD', 'admin')}@{os.getenv('MONGO_HOST', 'localhost')}:{os.getenv('MONGO_PORT', '27017')}/{os.getenv('MONGO_DATABASE', 'quants_lab')}"
mongo_client = MongoClient(uri)

connector_name = "okx_perpetual"
CONNECTOR_INSTANCE = clob.get_connector(connector_name)

# CLOBDataSource 
config['base_trading_pair'] = config["trading_pair"]
# 'grid_config_base'
prices = await utils.get_executor_prices(executor_config_dict = config, connector_instance = CONNECTOR_INSTANCE, side = 'long')
prices

KeyError: 'CRV-USDT'

In [ ]:
selected_config = all_configs[selected_controller]["config"]
selected_config

{'controller_name': 'stat_arb',
 'controller_type': 'generic',
 'total_amount_quote': 1000.0,
 'manual_kill_switch': None,
 'candles_config': [],
 'coerce_tp_to_step': True,
 'connector_name': 'okx_perpetual',
 'base_trading_pair': 'NOT-USDT',
 'quote_trading_pair': 'AIXBT-USDT',
 'base_side': 1,
 'grid_config_base': {'start_price': 0.002483396433152202,
  'end_price': 0.0028626035668477975,
  'limit_price': 0.0023885946497283037,
  'min_order_amount_quote': 3.024,
  'order_frequency': 5},
 'grid_config_quote': {'start_price': 0.1839879388208901,
  'end_price': 0.2092720611791099,
  'limit_price': 0.21559309176866487,
  'min_order_amount_quote': 3.024,
  'order_frequency': 5},
 'leverage': 50,
 'position_mode': 'HEDGE',
 'min_spread_between_orders': 4e-06,
 'max_open_orders': 3,
 'max_orders_per_batch': 1,
 'activation_bounds': 0.0003,
 'safe_extra_spread': 0.0002,
 'deduct_base_fees': False,
 'triple_barrier_config': {'stop_loss': 0.1,
  'take_profit': 0.008,
  'time_limit': 259200,
 

## Calculate overall metrics

In [ ]:
def calculate_metrics(df: pd.DataFrame, controller_config: Dict[str, Any], side: int = 1):
    side_key = "grid_config_base" if side == 1 else "grid_config_quote"
    total_amount_quote = controller_config["total_amount_quote"]
    global_pnl = df["global_pnl"].iloc[-1]
    max_draw_down = df["global_pnl"].min() / total_amount_quote
    max_run_up = df["global_pnl"].max() / total_amount_quote
    total_trades = len(df)
    total_quote_volume = df["quote_amount"].sum()
    total_duration_minutes = (df["timestamp"].max() - df["timestamp"].min()) / 60
    metrics = {
        "global_pnl": global_pnl,
        "max_draw_down": max_draw_down,
        "max_run_up": max_run_up,
        "total_trades": total_trades,
        "total_quote_volume": total_quote_volume,
        "total_duration_minutes": total_duration_minutes
    }
    return metrics

In [ ]:
calculate_metrics(long_df, selected_config, side=1)

{'global_pnl': 11.186988189999846,
 'max_draw_down': -0.005177015060000054,
 'max_run_up': 0.012384873189999887,
 'total_trades': 185,
 'total_quote_volume': 795.7900999999999,
 'total_duration_minutes': 126.38413333098093}

In [ ]:
calculate_metrics(short_df, selected_config, side=2)

{'global_pnl': 11.060134699999773,
 'max_draw_down': -0.00114397880000025,
 'max_run_up': 0.011177559699999782,
 'total_trades': 159,
 'total_quote_volume': 741.148,
 'total_duration_minutes': 80.26603333155315}

In [ ]:
long_df

,db_name,controller_id,side,trading_pair,order_type,trade_type,cumulative_fee_paid_quote,position_action,timestamp,price,...,base_amount_close,cum_base_open,cum_base_close,cum_quote_open,cum_quote_close,break_even_open,break_even_close,realized_pnl,unrealized_pnl,global_pnl
59,okx_perpetual-ts-9-02-2025.sqlite,okx_perpetual||NOT-USDT||AIXBT-USDT||2025||iso...,1,NOT-USDT,LIMIT,BUY,0.0017121,OPEN,1740575273.17700005,0.002634,...,0,1300,0,3.4242,0,0.002634,NaN,NaN,0,NaN
61,okx_perpetual-ts-9-02-2025.sqlite,okx_perpetual||NOT-USDT||AIXBT-USDT||2025||iso...,1,NOT-USDT,LIMIT,BUY,0.00068484,OPEN,1740575291.75,0.002634,...,0,2600,0,6.8484,0,0.002634,NaN,NaN,0,NaN
65,okx_perpetual-ts-9-02-2025.sqlite,okx_perpetual||NOT-USDT||AIXBT-USDT||2025||iso...,1,NOT-USDT,LIMIT,BUY,0.00063288,OPEN,1740575398.36199999,0.002637,...,0,3800,0,10.0128,0,0.00263495,NaN,NaN,0.0078,NaN
63,okx_perpetual-ts-9-02-2025.sqlite,okx_perpetual||NOT-USDT||AIXBT-USDT||2025||iso...,1,NOT-USDT,LIMIT,BUY,0.00063312,OPEN,1740575409.56999993,0.002638,...,0,5000,0,13.1784,0,0.00263568,NaN,NaN,0.0116,NaN
67,okx_perpetual-ts-9-02-2025.sqlite,okx_perpetual||NOT-USDT||AIXBT-USDT||2025||iso...,1,NOT-USDT,LIMIT,BUY,0.00063264,OPEN,1740575434.26900005,0.002636,...,0,6200,0,16.3416,0,0.00263574,NaN,NaN,0.0016,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,okx_perpetual-ts-9-02-2025.sqlite,okx_perpetual||NOT-USDT||AIXBT-USDT||2025||iso...,1,NOT-USDT,MARKET,SELL,0.00176995,CLOSE,1740582734.85299993,0.002723,...,1300,150200,69600,392.1594,184.7705,0.00261091,0.00265475,3.05083129,9.03406871,11.91001829
118,okx_perpetual-ts-9-02-2025.sqlite,okx_perpetual||NOT-USDT||AIXBT-USDT||2025||iso...,1,NOT-USDT,MARKET,SELL,0.00177125,CLOSE,1740582756.88899994,0.002725,...,1300,150200,70900,392.1594,188.313,0.00261091,0.00265604,3.19914208,9.04695792,12.06944704
120,okx_perpetual-ts-9-02-2025.sqlite,okx_perpetual||NOT-USDT||AIXBT-USDT||2025||iso...,1,NOT-USDT,MARKET,SELL,0.00177385,CLOSE,1740582783.26900005,0.002729,...,1300,150200,72200,392.1594,191.8607,0.00261091,0.00265735,3.35265286,9.21064714,12.38487319
183,okx_perpetual-ts-9-02-2025.sqlite,okx_perpetual||NOT-USDT||AIXBT-USDT||2025||iso...,1,NOT-USDT,MARKET,SELL,0.004887,CLOSE,1740582856.22399998,0.002715,...,3600,150200,75800,392.1594,201.6347,0.00261091,0.00266009,3.72735965,7.74394035,11.28798619


In [41]:
from core.data_sources.clob import CLOBDataSource
clob = CLOBDataSource()
from core.services.mongodb_client import MongoClient
from core.services.backend_api_client import BackendAPIClient

# mongo_client = MongoClient(
#     username=os.getenv("MONGO_INITDB_ROOT_USERNAME", "admin"),
#     password=os.getenv("MONGO_INITDB_ROOT_PASSWORD", "admin"),
#     host=os.getenv("MONGO_HOST", "localhost"),
#     port=os.getenv("MONGO_PORT", 27017),
#     database=os.getenv("MONGO_DATABASE", "quants_lab")
# )

uri = f"mongodb://{os.getenv('MONGO_INITDB_ROOT_USERNAME', 'admin')}:{os.getenv('MONGO_INITDB_ROOT_PASSWORD', 'admin')}@{os.getenv('MONGO_HOST', 'localhost')}:{os.getenv('MONGO_PORT', '27017')}/{os.getenv('MONGO_DATABASE', 'quants_lab')}"
mongo_client = MongoClient(uri)

connector_name = "okx_perpetual"
CONNECTOR_INSTANCE = clob.get_connector(connector_name)

# CLOBDataSource 
config['base_trading_pair'] = config["trading_pair"]
# 'grid_config_base'
prices = await utils.get_executor_prices(executor_config_dict = config, connector_instance = CONNECTOR_INSTANCE, side = 'long')
prices

KeyError: 'CRV-USDT'